# LLM Response Evaluation Guide

Evaluating Large Language Model (LLM) outputs is crucial for building reliable applications. Unlike traditional software, natural language outputs can't always be verified with simple assertions. This notebook demonstrates three major approaches to evaluation:

1. **Rule-Based/Heuristic Evaluation** (Deterministic metrics like length, keyword matching, and regex).
2. **Semantic Similarity Evaluation** (Using embeddings to compare semantic meaning instead of exact words).
3. **LLM-as-a-Judge** (Leveraging a powerful model to grade responses based on complex criteria, rubrics, or reference answers).

In [ ]:
# Install necessary libraries
!pip install google-genai numpy scikit-learn -q

## Setup and Configuration

First, we initialize the Gemini API client using the recommended `google-genai` SDK.

In [ ]:
import os
from google import genai
from google.colab import userdata

# Initialize the Gemini client
# Make sure GOOGLE_API_KEY is configured in your Colab secrets (🔑 icon on the left panel)
try:
    api_key = userdata.get('gemini_key')
    client = genai.Client(api_key=api_key)
    print("Gemini API client initialized successfully!")
except Exception as e:
    print(f"Error initializing Gemini client: {e}\nPlease ensure your GOOGLE_API_KEY is set in Colab secrets.")

Gemini API client initialized successfully!


## 1. Rule-Based / Heuristic Evaluation

These are fast, cheap, deterministic checks. They are great for guarding against specific failure modes (e.g., responses being too long, containing forbidden words, or failing to output valid JSON).

In [ ]:
import re

def evaluate_heuristics(response_text, forbidden_words=None, max_length=500):
    if forbidden_words is None:
        forbidden_words = []

    results = {}

    # Length check
    results['length_ok'] = len(response_text) <= max_length
    results['char_count'] = len(response_text)

    # Forbidden words check
    found_forbidden = [word for word in forbidden_words if word.lower() in response_text.lower()]
    results['no_forbidden_words'] = len(found_forbidden) == 0
    results['found_forbidden_words'] = found_forbidden

    # Structure check: Does it look like it has bullet points or structured text?
    results['is_bulleted'] = bool(re.search(r'^\s*[-*•]\s+', response_text, re.MULTILINE))

    return results

# Example usage. Change the text and see results
sample_output = """- First, gather all necessary ingredients including flour, sugar, and yeast.
- Second, mix them well in a bowl.
- Avoid using cold water to prevent slowing down the yeast activation."""

metrics = evaluate_heuristics(sample_output, forbidden_words=["easy", "simple"], max_length=200)
print("Heuristic Evaluation Results:")
for metric, value in metrics.items():
    print(f"  {metric}: {value}")

Heuristic Evaluation Results:
  length_ok: True
  char_count: 176
  no_forbidden_words: True
  found_forbidden_words: []
  is_bulleted: False


## 2. Semantic Similarity Evaluation

Sometimes a generated answer uses different words but shares the same meaning as a reference ground-truth answer. We can measure this by embedding both texts and computing their cosine similarity.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_embedding(text, model="gemini-embedding-001"):
    response = client.models.embed_content(
        model=model,
        contents=text
    )
    # The SDK returns the embedding values
    return response.embeddings[0].values

def evaluate_semantic_similarity(candidate_text, reference_text):
    emb_candidate = np.array(get_embedding(candidate_text)).reshape(1, -1)
    emb_reference = np.array(get_embedding(reference_text)).reshape(1, -1)

    similarity = cosine_similarity(emb_candidate, emb_reference)[0][0]
    return float(similarity)

# Example usage
reference = "The capital of France is Paris, famous for its culture and art."
candidate_good = "Paris is France's capital city, known globally as a hub for art and culture."
candidate_bad = "London is the capital of the United Kingdom, famous for its history."

sim_good = evaluate_semantic_similarity(candidate_good, reference)
sim_bad = evaluate_semantic_similarity(candidate_bad, reference)

print(f"Reference: '{reference}'")
print(f"Good Match: '{candidate_good}' | Similarity: {sim_good:.4f}")
print(f"Bad Match:  '{candidate_bad}' | Similarity: {sim_bad:.4f}")

Reference: 'The capital of France is Paris, famous for its culture and art.'
Good Match: 'Paris is France's capital city, known globally as a hub for art and culture.' | Similarity: 0.8222
Bad Match:  'London is the capital of the United Kingdom, famous for its history.' | Similarity: 0.7257


## 3. LLM-as-a-Judge

For complex aspects like tone, helpfulness, accuracy, or adherence to formatting instructions, using an LLM to evaluate another LLM is highly effective. Here, we define an evaluation rubric and ask Gemini to act as a judge, outputting its decision in a structured JSON format.

In [ ]:
from pydantic import BaseModel, Field

# Define the structured output format for our judge
class EvaluationScore(BaseModel):
    score: int = Field(description="A score from 1 to 5, where 5 is excellent and 1 is poor.")
    reasoning: str = Field(description="Detailed explanation justifying the given score based on the rubric.")
    critical_feedback: str = Field(description="Constructive feedback on what could be improved.")

def llm_judge_evaluate(prompt, response, rubric):
    system_instruction = (
        "You are an expert quality assurance judge evaluating LLM generated responses. "
        "Be objective, critical, and adhere strictly to the provided evaluation rubric."
    )

    user_content = f"""Please evaluate the response based on the original user prompt and the evaluation rubric.

[Original User Prompt]
{prompt}

[LLM Generated Response to Evaluate]
{response}

[Evaluation Rubric]
{rubric}
"""

    # Using gemini-2.5-flash for fast and smart evaluation with Structured Outputs
    eval_response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=user_content,
        config={
            "system_instruction": system_instruction,
            "response_mime_type": "application/json",
            "response_schema": EvaluationScore,
            "temperature": 0.1,  # Low temperature for more deterministic/consistent grading
        }
    )

    # Parse and return parsed structure directly
    return eval_response.parsed

# Let's test the LLM Judge
prompt = "Explain quantum computing to a 10-year-old child."

#Assume we got the below responses from an LLM
good_response = '''Imagine a normal computer is like a
light switch that can only be turned ON or OFF, but a quantum computer
uses qubits which are like spinning coins that can be both ON and OFF at
the same time! This special superpower lets quantum computers explore many
different solutions all at once, helping them solve complex puzzles incredibly fast.'''

bad_response = '''Quantum computing is a type of computation
whose operations can harness the phenomena of quantum mechanics,
such as superposition, interference, and entanglement.
Devices that perform quantum computations are known as quantum computers.'''

rubric = """
1. Tone and Vocabulary: Is it appropriate for a 10-year-old? Avoid overly academic/dense jargon without clear analogies.
2. Accuracy: Is the definition of quantum computing fundamentally correct?
3. Engagement: Is the explanation fun and easy to visualize?
"""

print("Evaluating Response...")
evaluation = llm_judge_evaluate(prompt, good_response, rubric)

print(f"\nScore: {evaluation.score}/5")
print(f"Reasoning:\n{evaluation.reasoning}")
print(f"\nFeedback for improvement:\n{evaluation.critical_feedback}")

Evaluating Response...

Score: 5/5
Reasoning:
The response is exceptionally well-suited for a 10-year-old. It uses clear, age-appropriate language and avoids dense academic jargon while maintaining fundamental scientific accuracy (superposition is explained using the spinning coin analogy). It is highly engaging, fun, and easy to visualize.

Feedback for improvement:
The explanation is concise and effective. No major improvements are needed, though adding a brief real-world example of what it solves could make it even more relatable.
